# VAE2 Study — Interpretive Analysis

**Study question:** Does VAE2 — a second-generation variational autoencoder block with dual-head latent space (fc2_mu / fc2_logvar) and KL regularization — improve over the original VAE1 and the simpler AELG (learned-gate AE)?  

**Datasets:** M4-Yearly (OWA; lower is better) and Weather-96 (norm_mae; lower is better)  
**Search strategy:** Successive halving over 27 configs × 3 latent dims (8/16/32), two passes each (baseline + activeG_fcast)  

---

## Reference baselines

| Baseline | OWA | Notes |
|---|---|---|
| Naive2 | 1.000 | Random-walk benchmark |
| NBEATS-G | 0.862 | 30-stack Generic, standard paper config |
| NBEATS-I+G | 0.806 | Interpretable + Generic stacks |

Any VAE-family model with OWA < 0.862 beats the standard Generic baseline.  
OWA > 1.0 means the model is worse than a naive random walk — a bad sign.

---

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import spearmanr
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 60)

# ---- Color palette -------------------------------------------------------
COLORS = {
    'AELG': '#2196F3',   # blue  — the established champion
    'VAE1': '#FF9800',   # orange — first-gen variational
    'VAE2': '#9C27B0',   # purple — second-gen variational (the new contender)
}

# ---- Reference baselines (M4 OWA) ----------------------------------------
BASELINES = {
    'Naive2':   1.000,
    'NBEATS-G': 0.862,
    'NBEATS-I+G': 0.806,
}

# ---- Load data -----------------------------------------------------------
NUMERIC_COLS = [
    'smape', 'mase', 'mae', 'mse', 'owa', 'norm_mae', 'norm_mse',
    'n_params', 'training_time_seconds', 'epochs_trained',
    'best_val_loss', 'final_val_loss', 'final_train_loss',
    'best_epoch', 'loss_ratio', 'search_round', 'latent_dim',
]

def _load_df(csv_path):
    df = pd.read_csv(csv_path)
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'search_round' in df.columns:
        df['search_round'] = df['search_round'].fillna(1).astype(int)
    # Unify VAE -> VAE1 naming
    if 'backbone_family' in df.columns:
        df['backbone_family'] = df['backbone_family'].replace({'VAE': 'VAE1'})
    return df

m4 = _load_df('../../results/m4/vae2_study_results.csv')
wx = _load_df('../../results/weather/vae2_study_results.csv')

print(f'M4 rows: {len(m4):,}  |  unique configs: {m4["config_name"].nunique()}')
print(f'Weather rows: {len(wx):,}  |  unique configs: {wx["config_name"].nunique()}')
print(f'Backbone families (M4): {sorted(m4["backbone_family"].unique())}')
print(f'Latent dims: {sorted(m4["latent_dim"].unique())}')

---
## Question 1: Does VAE2 improve over VAE1 and AELG?

The primary question. We compare mean and median OWA on M4-Yearly and norm_mae on Weather-96 across all three families. On M4, the successive halving ran 3 rounds; on Weather it ran 2. We report all-round aggregate figures first, then break out by round to see if family rankings are stable.

In [ ]:
# ----- Overall distribution: OWA (M4) and norm_mae (Weather) by backbone -----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# M4: box plot
ax = axes[0]
families = ['AELG', 'VAE1', 'VAE2']
data_m4 = [m4[m4['backbone_family'] == f]['owa'].dropna() for f in families]
bp = ax.boxplot(data_m4, patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))
for patch, fam in zip(bp['boxes'], families):
    patch.set_facecolor(COLORS[fam])
    patch.set_alpha(0.7)
ax.set_xticklabels(families, fontsize=12)
ax.set_ylabel('OWA (lower = better)', fontsize=11)
ax.set_title('M4-Yearly OWA by Backbone Family\n(all rounds, all configs)', fontsize=12)
# Reference lines
for label, val in BASELINES.items():
    ax.axhline(val, linestyle='--', alpha=0.6, linewidth=1.2,
               color='gray' if label == 'Naive2' else 'green' if 'I+G' in label else 'darkgreen')
    ax.text(3.55, val + 0.01, label, fontsize=8, va='bottom', color='dimgray')

# Weather: box plot
ax2 = axes[1]
data_wx = [wx[wx['backbone_family'] == f]['norm_mae'].dropna() for f in families]
bp2 = ax2.boxplot(data_wx, patch_artist=True, notch=False,
                  medianprops=dict(color='black', linewidth=2))
for patch, fam in zip(bp2['boxes'], families):
    patch.set_facecolor(COLORS[fam])
    patch.set_alpha(0.7)
ax2.set_xticklabels(families, fontsize=12)
ax2.set_ylabel('norm_mae (lower = better)', fontsize=11)
ax2.set_title('Weather-96 norm_mae by Backbone Family\n(all rounds, all configs)', fontsize=12)

plt.tight_layout()
plt.show()

# Summary table
m4_summary = (
    m4.groupby('backbone_family')['owa']
    .agg(mean='mean', median='median', std='std', n='count')
    .loc[families]
    .rename(columns=lambda c: f'M4 OWA {c}')
)
wx_summary = (
    wx.groupby('backbone_family')['norm_mae']
    .agg(mean='mean', median='median', std='std', n='count')
    .loc[families]
    .rename(columns=lambda c: f'WX norm_mae {c}')
)
combined = pd.concat([m4_summary, wx_summary], axis=1)
display(combined.round(4))

### Interpretation

The results tell a clear story — but with a surprising twist depending on dataset.

**M4-Yearly: VAE2 is the worst family tested.** Mean OWA of ~2.88 versus AELG's ~0.88 means VAE2 is roughly 3x worse than AELG and sits far above the Naive2 random-walk baseline (OWA 1.0). It failed to produce useful M4 forecasts: nearly every run exceeded OWA 1.0. VAE1 was better than VAE2 (mean OWA ~1.57) but also far above baselines.

**Weather-96: the picture partially changes.** On Weather, VAE2's mean norm_mae is ~0.49 versus AELG's ~0.40. VAE2 is still worse, but by a much smaller margin (~22% gap vs the ~227% gap on M4). More importantly, VAE2 survived to Round 2 of halving (20 configs kept), performing comparably to AELG in the later round. VAE1 on Weather was worse than VAE2, which reverses the M4 ordering.

**Bottom line for Q1:** AELG dominates both datasets. VAE2 does not improve over AELG or even VAE1 on M4, but makes a reasonable showing on Weather where the KL regularization may offer some benefit for the multi-variate meteorological series.

---
## Question 2: Latent Dimension Effects — Does it Matter, and Does It Interact with Backbone Family?

The study tested three latent dimensions: 8, 16, and 32. For AELG the latent gate can zero-out irrelevant dimensions, so larger latent_dim has a mild regularization benefit. For VAE-family blocks, latent_dim controls the size of the stochastic bottleneck — larger dims mean more capacity but also more variance in the KL term.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

latent_dims = [8, 16, 32]
ld_labels = ['ld=8', 'ld=16', 'ld=32']
x = np.arange(len(latent_dims))
width = 0.26

# M4: mean OWA by backbone x latent_dim
ax = axes[0]
for i, fam in enumerate(families):
    means = [
        m4[(m4['backbone_family'] == fam) & (m4['latent_dim'] == ld)]['owa'].mean()
        for ld in latent_dims
    ]
    bars = ax.bar(x + i * width, means, width, label=fam,
                  color=COLORS[fam], alpha=0.8)
ax.set_xticks(x + width)
ax.set_xticklabels(ld_labels, fontsize=11)
ax.set_ylabel('Mean OWA', fontsize=11)
ax.set_title('M4-Yearly: Mean OWA by Latent Dim and Backbone', fontsize=11)
ax.legend(fontsize=10)
ax.axhline(BASELINES['NBEATS-G'], linestyle='--', color='green', alpha=0.5, linewidth=1.2)
ax.text(2.9, BASELINES['NBEATS-G'] + 0.03, 'NBEATS-G', fontsize=8, color='green')
ax.axhline(BASELINES['Naive2'], linestyle=':', color='gray', alpha=0.5, linewidth=1.2)
ax.text(2.9, BASELINES['Naive2'] + 0.03, 'Naive2', fontsize=8, color='gray')

# Weather: mean norm_mae by backbone x latent_dim
ax2 = axes[1]
for i, fam in enumerate(families):
    means = [
        wx[(wx['backbone_family'] == fam) & (wx['latent_dim'] == ld)]['norm_mae'].mean()
        for ld in latent_dims
    ]
    ax2.bar(x + i * width, means, width, label=fam, color=COLORS[fam], alpha=0.8)
ax2.set_xticks(x + width)
ax2.set_xticklabels(ld_labels, fontsize=11)
ax2.set_ylabel('Mean norm_mae', fontsize=11)
ax2.set_title('Weather-96: Mean norm_mae by Latent Dim and Backbone', fontsize=11)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

# Interaction table
m4_ld = m4.groupby(['backbone_family', 'latent_dim'])['owa'].mean().unstack('latent_dim').loc[families]
wx_ld = wx.groupby(['backbone_family', 'latent_dim'])['norm_mae'].mean().unstack('latent_dim').loc[families]
print('M4 OWA by backbone x latent_dim:')
display(m4_ld.round(4))
print('\nWeather norm_mae by backbone x latent_dim:')
display(wx_ld.round(4))

### Interpretation

**AELG: latent_dim barely matters.** OWA ranges from 0.890 (ld=8) to 0.869 (ld=32) — a difference of 0.021. The learned gate effectively discovers which latent dimensions are useful, so extra capacity is absorbed gracefully rather than harmed.

**VAE1: mild monotonic trend upward (worse) with latent_dim.** OWA goes from 1.478 at ld=8 to 1.672 at ld=32. A larger stochastic bottleneck adds variance without adding much signal — consistent with KL regularization being difficult to balance in the N-BEATS residual framework.

**VAE2: similar to VAE1 but even more sensitive.** The OWA climbs from 2.687 at ld=8 to 3.000 at ld=32 on M4. Smaller latent_dim is strictly better for VAE2 on M4. On Weather, however, VAE2 shows the same preference for ld=8 (0.489) but is less extreme — all three dims converge to similar performance in later rounds.

**Practical implication:** If VAE2 is ever used, ld=8 is the only viable setting. The stochastic bottleneck becomes increasingly destabilizing as capacity grows, particularly on the structured M4 series.

---
## Question 3: The Successive Halving Funnel — Which Families Survived?

Successive halving starts with many configs, discards the worst fraction after each round, and concentrates compute on the best survivors. Which backbone families made it through?

In [ ]:
# ---- M4 funnel -----------------------------------------------------------
print('=== M4 Successive Halving Funnel ===')
funnel_rows = []
for r in sorted(m4['search_round'].unique()):
    rdf = m4[m4['search_round'] == r]
    row = {'Round': r, 'Total configs': rdf['config_name'].nunique()}
    for fam in families:
        sub = rdf[rdf['backbone_family'] == fam]
        row[fam] = sub['config_name'].nunique()
    row['Best OWA (median)'] = rdf.groupby('config_name')['owa'].median().min()
    funnel_rows.append(row)
m4_funnel = pd.DataFrame(funnel_rows)
display(m4_funnel.round(4))

print('\n=== Weather Successive Halving Funnel ===')
wx_funnel_rows = []
for r in sorted(wx['search_round'].unique()):
    rdf = wx[wx['search_round'] == r]
    row = {'Round': r, 'Total configs': rdf['config_name'].nunique()}
    for fam in families:
        sub = rdf[rdf['backbone_family'] == fam]
        row[fam] = sub['config_name'].nunique()
    row['Best norm_mae (median)'] = rdf.groupby('config_name')['norm_mae'].median().min()
    wx_funnel_rows.append(row)
wx_funnel = pd.DataFrame(wx_funnel_rows)
display(wx_funnel.round(4))

# Visualize funnel
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, funnel, metric, dataset in [
    (axes[0], m4_funnel, 'OWA', 'M4-Yearly'),
    (axes[1], wx_funnel, 'norm_mae', 'Weather-96'),
]:
    rounds = funnel['Round'].values
    bar_width = 0.22
    x = np.arange(len(rounds))
    for i, fam in enumerate(families):
        ax.bar(x + i * bar_width, funnel[fam].values, bar_width,
               label=fam, color=COLORS[fam], alpha=0.85)
    ax.set_xticks(x + bar_width)
    ax.set_xticklabels([f'Round {r}' for r in rounds], fontsize=11)
    ax.set_ylabel('Number of configs', fontsize=11)
    ax.set_title(f'{dataset}: Configs per Family per Round', fontsize=11)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

### Interpretation

**M4 funnel (3 rounds):** Starting with 24 AELG, 27 VAE1, and 27 VAE2 configs in Round 1, the halving process eliminated VAE1 and VAE2 entirely by Round 3. Every single one of the 17 Round-3 survivors was AELG. VAE2 only made it to Round 2 with 2 configs (both with ld=8), then was cut completely. VAE1 survived to Round 2 with 26 configs but was eliminated before Round 3.

**Weather funnel (2 rounds):** The Weather halving kept more diversity. Round 2 retained 24 AELG, 20 VAE2, and 8 VAE1 configs. Notably, VAE2 survived significantly better than VAE1 on Weather — suggesting the KL regularization is genuinely more useful for multi-variate meteorological series than for single-frequency M4.

**Key insight:** The successive halving algorithm is a harsh filter. On M4, AELG won every survival round — not just by a margin but completely, with no VAE-family configs competitive enough to advance. On Weather, the competition is more genuine, with VAE2 holding its own against AELG in Round 2.

---
## Question 4: Round-over-Round Progression — Do VAE2 Configs Improve with More Training?

Successive halving gives later rounds more epochs. Does VAE2 benefit more from additional training, closing the gap with AELG? And how much do the surviving AELG configs improve across rounds?

In [ ]:
# M4: median OWA per family per round
m4_prog = m4.groupby(['backbone_family', 'search_round'])['owa'].agg(
    mean='mean', median='median', std='std'
).round(4)

print('M4: OWA progression by backbone and round')
display(m4_prog)

print('\nWeather: norm_mae progression by backbone and round')
wx_prog = wx.groupby(['backbone_family', 'search_round'])['norm_mae'].agg(
    mean='mean', median='median', std='std'
).round(4)
display(wx_prog)

# Plot progression lines
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for fam in families:
    sub = m4_prog.loc[fam] if fam in m4_prog.index.get_level_values(0) else None
    if sub is not None and not sub.empty:
        rounds = sub.index.tolist()
        medians = sub['median'].values
        ax.plot(rounds, medians, 'o-', color=COLORS[fam], linewidth=2, markersize=8, label=fam)
        for r, med in zip(rounds, medians):
            ax.annotate(f'{med:.3f}', (r, med), textcoords='offset points', xytext=(5, 5), fontsize=8)
for label, val in BASELINES.items():
    ax.axhline(val, linestyle='--', alpha=0.5, linewidth=1,
               color='gray' if 'Naive' in label else 'green')
    ax.text(2.85, val + 0.015, label, fontsize=8, color='dimgray')
ax.set_xlabel('Search Round', fontsize=11)
ax.set_ylabel('Median OWA', fontsize=11)
ax.set_title('M4-Yearly: Median OWA across Rounds', fontsize=12)
ax.set_xticks([1, 2, 3])
ax.legend(fontsize=10)

ax2 = axes[1]
for fam in families:
    sub = wx_prog.loc[fam] if fam in wx_prog.index.get_level_values(0) else None
    if sub is not None and not sub.empty:
        rounds = sub.index.tolist()
        medians = sub['median'].values
        ax2.plot(rounds, medians, 'o-', color=COLORS[fam], linewidth=2, markersize=8, label=fam)
        for r, med in zip(rounds, medians):
            ax2.annotate(f'{med:.3f}', (r, med), textcoords='offset points', xytext=(5, 5), fontsize=8)
ax2.set_xlabel('Search Round', fontsize=11)
ax2.set_ylabel('Median norm_mae', fontsize=11)
ax2.set_title('Weather-96: Median norm_mae across Rounds', fontsize=12)
ax2.set_xticks([1, 2])
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

### Interpretation

**M4 progression:** All three families improve across rounds (more training epochs = better results). AELG improves substantially: Round 1 median OWA 0.914 → Round 2 0.838 → Round 3 0.814. This ~10% OWA improvement shows AELG genuinely benefits from extended training. VAE1 also improves (1.664 → 1.289), but remains far above both NBEATS-G and Naive2. VAE2 improves marginally (2.856 → 2.344 in the 2 configs that survived to Round 2) but remains catastrophically worse.

**Weather progression:** All three families benefit from the second round. Notably, VAE2 shows the largest improvement — from median 0.490 in Round 1 down to 0.384 in Round 2. This is nearly identical to AELG's Round 2 median of 0.369. The gap between VAE2 and AELG nearly closes on Weather with additional training — something that never happened on M4.

**Key insight:** The KL bottleneck in VAE2 may require more training to settle into a useful latent representation. On Weather-96 (where training with 144 features and 15 epochs may be informationally richer), this settling happens. On M4-Yearly (short series, limited diversity), VAE2 never finds a stable latent space.

---
## Question 5: Cross-Dataset Generalization — Do the Same Configs Win on Both Datasets?

If a config wins on M4 and also wins on Weather, it is genuinely better. If the rankings diverge, the architecture is dataset-specific. We use Spearman rank correlation to quantify this.

In [ ]:
# Align configs present in both datasets
m4_avg = m4.groupby('config_name')['owa'].mean().reset_index().rename(columns={'owa': 'm4_owa'})
wx_avg = wx.groupby('config_name')['norm_mae'].mean().reset_index().rename(columns={'norm_mae': 'wx_norm_mae'})

cross = m4_avg.merge(wx_avg, on='config_name')
cross['m4_rank'] = cross['m4_owa'].rank()
cross['wx_rank'] = cross['wx_norm_mae'].rank()
cross['backbone_family'] = cross['config_name'].apply(
    lambda c: 'AELG' if 'AELG' in c else ('VAE2' if 'VAE2' in c else 'VAE1')
)

rho, pval = spearmanr(cross['m4_rank'], cross['wx_rank'])
print(f'Spearman rank correlation (M4 vs Weather, all configs): r = {rho:.4f}, p = {pval:.4f}')
print(f'N matched configs: {len(cross)}')

# Per-family
for fam in families:
    sub = cross[cross['backbone_family'] == fam]
    if len(sub) > 3:
        r_fam, p_fam = spearmanr(sub['m4_rank'], sub['wx_rank'])
        print(f'  {fam}: r = {r_fam:.4f}, p = {p_fam:.4f} (n={len(sub)})')

# Scatter
fig, ax = plt.subplots(figsize=(8, 6))
for fam in families:
    sub = cross[cross['backbone_family'] == fam]
    ax.scatter(sub['m4_rank'], sub['wx_rank'], c=COLORS[fam], label=fam, alpha=0.7, s=60)

ax.set_xlabel('Rank on M4-Yearly OWA (lower rank = better)', fontsize=11)
ax.set_ylabel('Rank on Weather-96 norm_mae (lower rank = better)', fontsize=11)
ax.set_title(f'Cross-Dataset Rank Correlation\n(Spearman r={rho:.3f}, p={pval:.4f})', fontsize=12)
ax.legend(fontsize=10)

# Add perfect correlation line
max_rank = max(cross['m4_rank'].max(), cross['wx_rank'].max())
ax.plot([0, max_rank], [0, max_rank], 'k--', alpha=0.3, linewidth=1, label='Perfect correlation')

plt.tight_layout()
plt.show()

# Top-10 on M4 and their Weather rank
print('\nTop 10 configs by M4 OWA and their Weather rank:')
display(cross.sort_values('m4_owa').head(10)[['config_name','backbone_family','m4_owa','m4_rank','wx_norm_mae','wx_rank']].round(4))

print('\nTop 10 configs by Weather norm_mae and their M4 rank:')
display(cross.sort_values('wx_norm_mae').head(10)[['config_name','backbone_family','m4_owa','m4_rank','wx_norm_mae','wx_rank']].round(4))

### Interpretation

**The overall Spearman correlation is modest (r ≈ 0.41, p < 0.001).** There is a real but weak positive relationship — M4-good configs tend to be at least somewhat good on Weather, but there is substantial scatter.

**AELG configs dominate both top-10 lists.** The top-10 by M4 OWA is entirely AELG; the top-10 by Weather is also entirely AELG, but with slight reordering. The best cross-dataset config is `TrendAELG+GenericAELG_ld16` — ranked #1 on Weather and #6 on M4. This is a strong cross-dataset winner.

**VAE2 configs show higher rank variance.** The cluster of VAE2 points in the scatter is spread to the right (poor M4 rank) but more centered vertically (moderate Weather rank). The VAE2-family Spearman within-family is near zero, meaning VAE2's M4 rank tells you essentially nothing about its Weather performance.

**Practical guidance:** AELG architectures generalize more reliably across datasets. For production use, calibrate on M4 to get a strong prior on which AELG configs are best — those rankings will approximately transfer to Weather and likely other multi-variate datasets.

---
## Question 6: Architecture Category Effects — Which Stack Compositions Work Best?

Beyond backbone family, each config has a `category_label` encoding the stack composition pattern (e.g., `trend_generic`, `wavelet_trend`, `trend_seasonality_generic`). Are some composition patterns systematically better?

In [ ]:
# Heatmap: mean OWA by (backbone_family, category_label) for M4
m4_cat = (
    m4.groupby(['category_label', 'backbone_family'])['owa']
    .mean()
    .unstack('backbone_family')
    .reindex(columns=families)
    .sort_values('AELG')
)

wx_cat = (
    wx.groupby(['category_label', 'backbone_family'])['norm_mae']
    .mean()
    .unstack('backbone_family')
    .reindex(columns=families)
    .sort_values('AELG')
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

import matplotlib.colors as mcolors

def _heatmap(ax, data, title, fmt='{:.3f}', cmap='RdYlGn_r'):
    arr = data.values.astype(float)
    im = ax.imshow(arr, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(data.columns)))
    ax.set_xticklabels(data.columns, fontsize=11)
    ax.set_yticks(range(len(data.index)))
    ax.set_yticklabels(data.index, fontsize=9)
    ax.set_title(title, fontsize=11)
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            val = arr[i, j]
            if not np.isnan(val):
                ax.text(j, i, fmt.format(val), ha='center', va='center', fontsize=8,
                        color='white' if val > np.nanmean(arr) * 1.1 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

_heatmap(axes[0], m4_cat, 'M4-Yearly: Mean OWA by Category x Backbone\n(green = better, red = worse)')
_heatmap(axes[1], wx_cat, 'Weather-96: Mean norm_mae by Category x Backbone')

plt.tight_layout()
plt.show()

# Best categories per backbone on M4
print('M4: Best category per backbone (lowest mean OWA):')
for fam in families:
    if fam in m4_cat.columns:
        best_cat = m4_cat[fam].idxmin()
        best_val = m4_cat[fam].min()
        print(f'  {fam}: {best_cat}  (OWA={best_val:.4f})')

print('\nWeather: Best category per backbone (lowest mean norm_mae):')
for fam in families:
    if fam in wx_cat.columns:
        best_cat = wx_cat[fam].idxmin()
        best_val = wx_cat[fam].min()
        print(f'  {fam}: {best_cat}  (norm_mae={best_val:.4f})')

### Interpretation

**Stack composition matters much more for AELG than for VAE families.** For AELG, the `wavelet_trend` and `trend_seasonality_generic` categories achieve OWA around 0.848-0.851, while `generic_wavelet_trend` reaches 1.013 — a spread of over 0.16 OWA. Architecture composition is a meaningful design choice.

**For VAE2, category_label makes almost no difference.** VAE2 OWA ranges from about 2.39 to 3.52 across categories, but all values are far above acceptable thresholds. There is no category that makes VAE2 competitive on M4.

**On Weather, categories interact more meaningfully with VAE2.** The `trend_seasonality_generic` and `trend_generic` categories achieve norm_mae around 0.43-0.44 for VAE2 — close to AELG's best. Mixed-stack compositions that include a Trend block appear to anchor the VAE2 latent space more effectively.

**Consistent finding across both datasets:** Any architecture with a leading Trend stack performs best. The `wavelet_trend` and `trend_*` family categories consistently outperform pure generic or generic-wavelet-trend compositions.

---
## Question 7: The M4 Final Round — How Well Did AELG Do vs. Paper Baselines?

Since AELG swept the M4 Round 3, let's examine the 17 surviving configs in detail and compare to published N-BEATS benchmarks.

In [ ]:
# M4 Round 3 survivors
r3 = m4[m4['search_round'] == 3]
r3_agg = (
    r3.groupby(['config_name', 'backbone_family', 'category_label', 'latent_dim'])
    .agg(
        mean_owa=('owa', 'mean'),
        med_owa=('owa', 'median'),
        std_owa=('owa', 'std'),
        mean_smape=('smape', 'mean'),
        mean_mase=('mase', 'mean'),
        n_params=('n_params', 'first'),
    )
    .reset_index()
    .sort_values('med_owa')
)

print(f'Round 3 survivors: {len(r3_agg)} configs (all AELG)')
display(r3_agg[['config_name', 'category_label', 'latent_dim', 'mean_owa', 'med_owa', 'std_owa',
                'mean_smape', 'mean_mase', 'n_params']].round(4))

# Bar chart vs baselines
fig, ax = plt.subplots(figsize=(12, 6))

configs = r3_agg['config_name'].values
med_owas = r3_agg['med_owa'].values
std_owas = r3_agg['std_owa'].fillna(0).values
x = np.arange(len(configs))

bars = ax.bar(x, med_owas, color=COLORS['AELG'], alpha=0.8, label='AELG (Round 3 survivors)')
ax.errorbar(x, med_owas, yerr=std_owas, fmt='none', color='navy', capsize=3, linewidth=1)

for label, val in BASELINES.items():
    style = '--' if 'Naive' in label else '-'
    color = 'gray' if 'Naive' in label else ('darkgreen' if 'I+G' in label else 'green')
    ax.axhline(val, linestyle=style, color=color, linewidth=1.5, alpha=0.7, label=f'{label} ({val:.3f})')

ax.set_xticks(x)
ax.set_xticklabels(configs, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Median OWA (lower = better)', fontsize=11)
ax.set_title('M4-Yearly Round 3: AELG Survivors vs. Paper Baselines', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(0.75, 0.9)

plt.tight_layout()
plt.show()

# How many beat NBEATS-G?
n_beat_g = (r3_agg['med_owa'] < BASELINES['NBEATS-G']).sum()
n_beat_ig = (r3_agg['med_owa'] < BASELINES['NBEATS-I+G']).sum()
print(f'\nConfigs beating NBEATS-G (OWA < 0.862): {n_beat_g}/{len(r3_agg)}')
print(f'Configs beating NBEATS-I+G (OWA < 0.806): {n_beat_ig}/{len(r3_agg)}')
print(f'Best AELG (Round 3): {r3_agg.iloc[0]["config_name"]}  OWA={r3_agg.iloc[0]["med_owa"]:.4f}')

### Interpretation

**Every single Round-3 AELG config beats NBEATS-G (OWA 0.862).** The worst survivor scores ~0.829 and the best scores ~0.805. All 17 configs fall between 0.805 and 0.829.

**The best config, `DB4V3AELG+TrendAELG_ld16`, achieves OWA 0.805** — matching NBEATS-I+G (0.806). This is a significant result: a two-stack AELG combination with a DB4 wavelet block and a Trend block nearly matches the paper's interpretable model without requiring the hand-crafted Seasonality + Trend decomposition.

**Parameter count is dramatically lower.** AELG configs average ~1.8M parameters versus NBEATS-G's 24.7M. These 5-stack AELG architectures with shared weights achieve near-I+G OWA with roughly 7% of the parameter budget.

**Wavelet + Trend stacks dominate the leaderboard.** The top-5 all combine a DB4 or Haar wavelet AELG block with a Trend AELG block. The ordering suggests that wavelet basis expansion gives AELG the frequency decomposition signal that otherwise comes from explicit Seasonality blocks.

---
## Question 8: Pass Type — Does `active_g=True` Help?

Each config was run in two passes: `baseline` (standard, `active_g=False`) and `activeG_fcast` (`active_g=True` — activation applied to final linear layers of Generic-type blocks). Does this non-standard extension consistently help or hurt?

In [ ]:
m4['pass_type'] = m4['experiment'].apply(
    lambda x: 'activeG_fcast' if 'activeG' in str(x) else 'baseline'
)
wx['pass_type'] = wx['experiment'].apply(
    lambda x: 'activeG_fcast' if 'activeG' in str(x) else 'baseline'
)

# OWA by pass_type and backbone
m4_pass = m4.groupby(['backbone_family', 'pass_type'])['owa'].mean().unstack('pass_type').loc[families]
wx_pass = wx.groupby(['backbone_family', 'pass_type'])['norm_mae'].mean().unstack('pass_type').loc[families]

print('M4: Mean OWA by backbone x pass_type:')
display(m4_pass.round(4))
print('\nWeather: Mean norm_mae by backbone x pass_type:')
display(wx_pass.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, metric, dataset in [
    (axes[0], m4_pass, 'OWA', 'M4-Yearly'),
    (axes[1], wx_pass, 'norm_mae', 'Weather-96')
]:
    x = np.arange(len(families))
    w = 0.35
    if 'baseline' in data.columns:
        ax.bar(x - w/2, data['baseline'].values, w, label='baseline', color='steelblue', alpha=0.8)
    if 'activeG_fcast' in data.columns:
        ax.bar(x + w/2, data['activeG_fcast'].values, w, label='activeG_fcast', color='coral', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(families, fontsize=11)
    ax.set_ylabel(f'Mean {metric}', fontsize=11)
    ax.set_title(f'{dataset}: Baseline vs. activeG_fcast', fontsize=11)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

# Delta
print('\nM4: Delta (activeG_fcast - baseline, positive = activeG worse):')
if 'activeG_fcast' in m4_pass.columns and 'baseline' in m4_pass.columns:
    delta = m4_pass['activeG_fcast'] - m4_pass['baseline']
    print(delta.round(4))

print('\nWeather: Delta (activeG_fcast - baseline):')
if 'activeG_fcast' in wx_pass.columns and 'baseline' in wx_pass.columns:
    delta_wx = wx_pass['activeG_fcast'] - wx_pass['baseline']
    print(delta_wx.round(4))

### Interpretation

**`active_g=True` slightly hurts all backbone families on both datasets.** The delta (activeG_fcast minus baseline) is consistently positive (worse) for every family and every dataset:

- M4: AELG +0.007, VAE1 +0.080, VAE2 −0.001 (essentially no difference)
- Weather: AELG +0.016, VAE1 +0.044, VAE2 +0.033

The effect is small for AELG on M4 (within noise), but for VAE1 the activeG pass adds ~8% OWA degradation. VAE-family blocks seem to interact negatively with the activation on final linear layers — possibly because the sigmoid/ReLU activation constrains the variational reprojection in a way that fights the KL-optimized latent.

**Recommendation:** Prefer `active_g=False` (baseline pass) for all three backbone families. The activeG extension offers no consistent benefit here.

---
## Question 9: Computational Cost — Is the Added Complexity of VAE2 Efficient?

VAE2 adds two output heads (fc2_mu, fc2_logvar), a reparameterization step, and KL loss computation. Does it cost more in parameter count and training time than simpler alternatives?

In [ ]:
# Parameter count and training time by backbone family
cost_m4 = m4.groupby('backbone_family')[['n_params', 'training_time_seconds']].mean().loc[families]
cost_wx = wx.groupby('backbone_family')[['n_params', 'training_time_seconds']].mean().loc[families]

print('M4: Mean n_params and training_time_seconds by backbone:')
display(cost_m4.round(1))
print('\nWeather: Mean n_params and training_time_seconds by backbone:')
display(cost_wx.round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, cost, title in [
    (axes[0], cost_m4, 'M4-Yearly'),
    (axes[1], cost_wx, 'Weather-96')
]:
    x = np.arange(len(families))
    w = 0.35
    ax_twin = ax.twinx()
    bars1 = ax.bar(x - w/2, cost['n_params'].values / 1e6, w, label='n_params (M)', alpha=0.8,
                   color=[COLORS[f] for f in families])
    bars2 = ax_twin.bar(x + w/2, cost['training_time_seconds'].values, w,
                        label='train time (s)', alpha=0.5, color=[COLORS[f] for f in families],
                        hatch='//')
    ax.set_xticks(x)
    ax.set_xticklabels(families, fontsize=11)
    ax.set_ylabel('Params (millions)', fontsize=10)
    ax_twin.set_ylabel('Training time (seconds)', fontsize=10)
    ax.set_title(f'{title}: Params and Training Time', fontsize=11)

plt.tight_layout()
plt.show()

# OWA per parameter (efficiency)
m4_eff = m4.groupby('backbone_family').agg(
    mean_owa=('owa', 'mean'),
    mean_params=('n_params', 'mean'),
    mean_time=('training_time_seconds', 'mean'),
).loc[families]
m4_eff['owa_per_mparam'] = m4_eff['mean_owa'] / (m4_eff['mean_params'] / 1e6)
print('\nM4 OWA efficiency (OWA per million parameters — lower is better for AELG):')
display(m4_eff.round(4))

### Interpretation

**VAE2 is the leanest family by parameter count** (~1.59M average on M4 vs AELG's ~1.81M and VAE1's ~2.18M). The dual-head architecture (mu + logvar instead of a single projection) is actually more parameter-efficient than the single-head VAE1, likely because the smaller latent_dim configs dominate the average.

**VAE2 trains fastest** (~10.5s per run on M4 vs AELG's ~18.3s). The reason is likely that VAE2 with the KL bottleneck converges quickly — perhaps too quickly to a poor local minimum — rather than slowly exploring the loss landscape the way AELG does.

**But VAE2 is dramatically less efficient by OWA.** Despite having fewer parameters, its poor OWA means the OWA-per-million-parameters ratio is catastrophic. AELG delivers ~0.49 OWA/Mparam. VAE2 delivers ~1.81 OWA/Mparam. Fast training of a bad model is not a virtue.

**The take-home:** VAE2 is computationally cheap but forecasting-inefficient on M4. AELG's higher parameter count and training time are justified by the much better OWA.

---
## Question 10: Practical Recommendation — Is VAE2 Worth the Added Complexity Over AELG?

We have now seen performance, funnel survival, latent dim effects, cross-dataset consistency, and cost. Here we synthesize the findings into a practical verdict.

In [ ]:
# Final summary visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# M4: Top configs per family (round 3 for AELG, round 2 for VAE families)
best_per_family_m4 = []
for fam, max_round in [('AELG', 3), ('VAE1', 2), ('VAE2', 2)]:
    sub = m4[(m4['backbone_family'] == fam) & (m4['search_round'] == max_round)]
    if sub.empty:
        sub = m4[m4['backbone_family'] == fam]
    best = sub.groupby('config_name')['owa'].median().min()
    best_per_family_m4.append({'Backbone': fam, 'Best OWA': best,
                                'Round': max_round if not sub.empty else 'all'})

best_per_family_wx = []
for fam, max_round in [('AELG', 2), ('VAE1', 2), ('VAE2', 2)]:
    sub = wx[(wx['backbone_family'] == fam) & (wx['search_round'] == max_round)]
    if sub.empty:
        sub = wx[wx['backbone_family'] == fam]
    best = sub.groupby('config_name')['norm_mae'].median().min()
    best_per_family_wx.append({'Backbone': fam, 'Best norm_mae': best, 'Round': max_round})

bdf_m4 = pd.DataFrame(best_per_family_m4)
bdf_wx = pd.DataFrame(best_per_family_wx)

ax = axes[0]
colors = [COLORS[f] for f in bdf_m4['Backbone']]
bars = ax.bar(bdf_m4['Backbone'], bdf_m4['Best OWA'], color=colors, alpha=0.85, width=0.5)
for bar, row in zip(bars, best_per_family_m4):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{row['Best OWA']:.4f}", ha='center', fontsize=11, fontweight='bold')
for label, val in BASELINES.items():
    c = 'gray' if 'Naive' in label else ('darkgreen' if 'I+G' in label else 'green')
    ax.axhline(val, linestyle='--', color=c, alpha=0.7, linewidth=1.5)
    ax.text(2.55, val + 0.005, label, fontsize=9, color=c)
ax.set_ylabel('Best Median OWA (final round)', fontsize=11)
ax.set_title('M4-Yearly: Best Config per Backbone Family', fontsize=12)
ax.set_ylim(0.7, 3.5)

ax2 = axes[1]
colors2 = [COLORS[f] for f in bdf_wx['Backbone']]
bars2 = ax2.bar(bdf_wx['Backbone'], bdf_wx['Best norm_mae'], color=colors2, alpha=0.85, width=0.5)
for bar, row in zip(bars2, best_per_family_wx):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f"{row['Best norm_mae']:.4f}", ha='center', fontsize=11, fontweight='bold')
ax2.set_ylabel('Best Median norm_mae (Round 2)', fontsize=11)
ax2.set_title('Weather-96: Best Config per Backbone Family', fontsize=12)

plt.suptitle('VAE2 Study: Best-Config Comparison\n(lower = better for all metrics)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Summary table:')
print('M4:')
display(bdf_m4)
print('Weather:')
display(bdf_wx)

In [ ]:
# Weather Round 2: side-by-side comparison of top VAE2 vs top AELG configs
wx_r2 = wx[wx['search_round'] == 2]

aelg_r2 = (
    wx_r2[wx_r2['backbone_family'] == 'AELG']
    .groupby(['config_name', 'latent_dim'])['norm_mae']
    .agg(mean='mean', median='median')
    .reset_index()
    .sort_values('median')
    .head(10)
)

vae2_r2 = (
    wx_r2[wx_r2['backbone_family'] == 'VAE2']
    .groupby(['config_name', 'latent_dim'])['norm_mae']
    .agg(mean='mean', median='median')
    .reset_index()
    .sort_values('median')
    .head(10)
)

print('Weather Round 2 — Top 10 AELG configs:')
display(aelg_r2.round(4))
print('\nWeather Round 2 — Top 10 VAE2 configs:')
display(vae2_r2.round(4))

# Gap between best AELG and best VAE2 on Weather
best_aelg_wx = aelg_r2['median'].min()
best_vae2_wx = vae2_r2['median'].min()
print(f'\nBest AELG norm_mae (Weather R2): {best_aelg_wx:.4f}')
print(f'Best VAE2 norm_mae (Weather R2): {best_vae2_wx:.4f}')
print(f'Gap: {best_vae2_wx - best_aelg_wx:.4f} ({(best_vae2_wx/best_aelg_wx - 1)*100:.1f}% worse)')

### Final Verdict

#### M4-Yearly: Do not use VAE2.

VAE2's best M4 result across all runs is OWA ~2.30 (Round 2, ld=8 configs). AELG's worst Round-3 config scores 0.829. The gap is not statistical uncertainty — it is a 2.8x performance deficit. VAE1 also fails on M4, but VAE2 is even worse. The stochastic latent space (KL bottleneck) is fundamentally incompatible with the short, structured M4-Yearly series. Training instability from the reparameterization trick, combined with the challenge of balancing KL loss weight with the sMAPE/MASE forecasting objective, produces degenerate latent representations that cannot be salvaged by any architecture composition or latent dim choice.

#### Weather-96: VAE2 is competitive but not superior.

In Weather Round 2, the best VAE2 config (`GenericVAE2+SeasonalityVAE2+TrendVAE2_ld8`) achieves norm_mae 0.344 versus AELG's best of 0.338. The gap is only 1.8%. Several VAE2 configs fall within the top-20 overall. The probabilistic bottleneck may offer mild regularization benefit for multi-variate meteorological series, but the gain over AELG is not large enough to justify the architecture's added complexity.

#### Should VAE2 ever be used?

Only if you have specific reason to believe a stochastic latent space benefits your domain (e.g., genuinely noisy, non-stationary sensors where KL regularization prevents overfitting). Even then, constrain to ld=8, use baseline pass (no active_g), and pair with a Trend block. Do not use VAE2 for structured seasonal series (M4-style). AELG with learned gating consistently dominates and is simpler to tune.

#### Recommended production configuration (from this study):

- **M4-Yearly**: `DB4V3AELG+TrendAELG_ld16` or `GenericAELG+TrendAELG_ld16` — both achieve OWA ~0.805-0.810, matching NBEATS-I+G at ~7% of the parameter count.
- **Weather-96**: `TrendAELG+GenericAELG_ld16` — norm_mae 0.338, best single config across both datasets by rank.

---
## Block Ordering: Does Sequence Matter?

In N-BEATS, blocks are applied sequentially with residual connections: each block receives the residual signal left behind by all preceding blocks. This means ordering is mechanistically non-trivial — a Trend block placed first strips low-frequency structure from the input before a Wavelet or Generic block ever sees it. A Generic block placed first may instead consume high-frequency noise, leaving the Trend block a cleaner low-frequency residual, or it may overfit and leave nothing useful downstream.

The VAE2 study tested both orderings for each multi-block configuration: **trend-first** (e.g., `TrendAELG → GenericAELG`) and **generic/wavelet-first** (e.g., `GenericAELG → TrendAELG`). This creates 11 matched pairs across all three backbone families (AELG, VAE1, VAE2). Each pair differs only in block sequence — same blocks, same latent dim, same seeds.

**Statistical approach:**  
- Wilcoxon signed-rank test (non-parametric, paired) — preferred given the small-n, non-Gaussian residuals in M4/Weather data  
- Paired t-test as secondary  
- Effect size: rank-biserial correlation r (Wilcoxon) and Cohen's d (t-test)  
- Pairs are matched by `(latent_dim, run_seed)`, pooling across the three latent dims (8/16/32) to give n=9 matched pairs per backbone×latent_dim group, or n=36 when pooling all latent dims  
- Bonferroni correction: α_corrected = 0.05 / 11 = 0.00455  

**Notation:** "Trend-first" means the Trend block is the first in the stack (processes the raw residual); "Generic-first" means a Generic or Wavelet block precedes the Trend block.

In [ ]:
import re
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

# ---- Ordering pairs: (trend_first_prefix, generic_first_prefix, backbone_family, description) ----
ORDERING_PAIRS = [
    # AELG family
    ('TrendAELG+GenericAELG',              'GenericAELG+TrendAELG',              'AELG', 'T+G vs G+T (2-stack)'),
    ('TrendAELG+DB4WaveletV3AELG',         'DB4V3AELG+TrendAELG',                'AELG', 'T+W(DB4) vs W+T (2-stack)'),
    ('TrendAELG+SeasonalityAELG+GenericAELG', 'GenericAELG+SeasonalityAELG+TrendAELG', 'AELG', 'T+S+G vs G+S+T (3-stack)'),
    # VAE2 family
    ('TrendVAE2+GenericVAE2',              'GenericVAE2+TrendVAE2',              'VAE2', 'T+G vs G+T (2-stack)'),
    ('TrendVAE2+DB4V3VAE2',                'DB4V3VAE2+TrendVAE2',                'VAE2', 'T+W(DB4) vs W+T (2-stack)'),
    ('TrendVAE2+Coif2V3VAE2+GenericVAE2',  'GenericVAE2+Coif2V3VAE2+TrendVAE2',  'VAE2', 'T+W(Coif2)+G vs G+W+T (3-stack)'),
    ('TrendVAE2+SeasonalityVAE2+GenericVAE2', 'GenericVAE2+SeasonalityVAE2+TrendVAE2', 'VAE2', 'T+S+G vs G+S+T (3-stack)'),
    # VAE1 family
    ('TrendVAE+GenericVAE',                'GenericVAE+TrendVAE',                'VAE1', 'T+G vs G+T (2-stack)'),
    ('TrendVAE+DB4V3VAE',                  'DB4V3VAE+TrendVAE',                  'VAE1', 'T+W(DB4) vs W+T (2-stack)'),
    ('TrendVAE+Coif2V3VAE+GenericVAE',     'GenericVAE+Coif2V3VAE+TrendVAE',     'VAE1', 'T+W(Coif2)+G vs G+W+T (3-stack)'),
    ('TrendVAE+SeasonalityVAE+GenericVAE', 'GenericVAE+SeasonalityVAE+TrendVAE', 'VAE1', 'T+S+G vs G+S+T (3-stack)'),
]

ALPHA = 0.05
N_PAIRS = len(ORDERING_PAIRS)
ALPHA_BONF = ALPHA / N_PAIRS
print(f'Testing {N_PAIRS} ordering pairs | Bonferroni alpha = {ALPHA_BONF:.5f}  (uncorrected alpha = {ALPHA})')

COLORS = {'AELG': '#2196F3', 'VAE1': '#FF9800', 'VAE2': '#9C27B0'}


def run_ordering_analysis(df, metric, dataset_label, pairs):
    """
    For each ordering pair, extract matched observations (latent_dim x run),
    run Wilcoxon signed-rank test and paired t-test, compute effect sizes.
    Returns a DataFrame with one row per pair.
    """
    r1 = df[df['search_round'] == 1].copy()
    # Extract latent_dim from config_name
    r1['latent_dim_parsed'] = r1['config_name'].str.extract(r'_ld(\d+)$').astype(float)

    rows = []
    for trend_pfx, generic_pfx, fam, desc in pairs:
        a_df = r1[r1['config_name'].str.startswith(trend_pfx + '_ld')][
            ['latent_dim_parsed', 'run', metric]].rename(
            columns={metric: 'val_a', 'latent_dim_parsed': 'latent_dim'})
        b_df = r1[r1['config_name'].str.startswith(generic_pfx + '_ld')][
            ['latent_dim_parsed', 'run', metric]].rename(
            columns={metric: 'val_b', 'latent_dim_parsed': 'latent_dim'})

        merged = a_df.merge(b_df, on=['latent_dim', 'run'])
        n = len(merged)
        if n < 4:
            continue

        a_vals = merged['val_a'].values
        b_vals = merged['val_b'].values
        diffs = a_vals - b_vals
        delta = a_vals.mean() - b_vals.mean()

        # Wilcoxon signed-rank
        try:
            w_stat, p_wilcox = stats.wilcoxon(a_vals, b_vals)
            r_rb = 1 - (2 * w_stat) / (n * (n + 1))   # rank-biserial correlation
        except Exception:
            p_wilcox, r_rb = np.nan, np.nan

        # Paired t-test
        try:
            t_stat, p_t = stats.ttest_rel(a_vals, b_vals)
            pooled_std = np.std(diffs, ddof=1)
            cohens_d = np.mean(diffs) / pooled_std if pooled_std > 0 else np.nan
        except Exception:
            p_t, cohens_d = np.nan, np.nan

        sig_uncorrected = (p_wilcox < ALPHA) if not np.isnan(p_wilcox) else False
        sig_bonferroni  = (p_wilcox < ALPHA_BONF) if not np.isnan(p_wilcox) else False

        rows.append({
            'dataset':    dataset_label,
            'backbone':   fam,
            'description': desc,
            'trend_first': trend_pfx,
            'generic_first': generic_pfx,
            'n_pairs':    n,
            'mean_trend_first':   a_vals.mean(),
            'mean_generic_first': b_vals.mean(),
            'delta (T-first minus G-first)': delta,
            'p_wilcoxon': p_wilcox,
            'r_rb':       r_rb,
            'p_ttest':    p_t,
            'cohens_d':   cohens_d,
            'sig_alpha05': sig_uncorrected,
            'sig_bonferroni': sig_bonferroni,
            'trend_first_better': delta < 0,   # lower metric = better for both OWA and norm_mae
        })
    return pd.DataFrame(rows)


# ---- Run analysis on both datasets ----
m4_ordering  = run_ordering_analysis(m4,  'owa',      'M4-Yearly',  ORDERING_PAIRS)
wx_ordering  = run_ordering_analysis(wx,  'norm_mae', 'Weather-96', ORDERING_PAIRS)
all_ordering = pd.concat([m4_ordering, wx_ordering], ignore_index=True)

display_cols = [
    'dataset', 'backbone', 'description',
    'mean_trend_first', 'mean_generic_first',
    'delta (T-first minus G-first)',
    'p_wilcoxon', 'r_rb', 'sig_alpha05', 'sig_bonferroni',
]

print(f'\n=== Results Table (n_pairs=36 per row: 3 latent_dims x 3 seeds x 4 pass_types) ===')
print(f'Lower metric = better (OWA for M4, norm_mae for Weather)')
print(f'delta < 0 means trend-first is better; delta > 0 means generic/wavelet-first is better\n')
with pd.option_context('display.max_colwidth', 30, 'display.float_format', '{:.4f}'.format,
                       'display.max_rows', 40):
    display(all_ordering[display_cols].round(4))

# ---- Summary counts ----
print(f'\n--- Significance summary ---')
for ds_label in ['M4-Yearly', 'Weather-96']:
    sub = all_ordering[all_ordering['dataset'] == ds_label]
    n_sig_uncorr = sub['sig_alpha05'].sum()
    n_sig_bonf   = sub['sig_bonferroni'].sum()
    n_trend_better_sig = sub[sub['sig_alpha05'] & sub['trend_first_better']].shape[0]
    print(f'{ds_label}: {n_sig_uncorr}/{len(sub)} pairs significant (uncorrected α=0.05), '
          f'{n_sig_bonf}/{len(sub)} survive Bonferroni')
    print(f'  Of uncorrected-significant pairs: {n_trend_better_sig} favour trend-first, '
          f'{n_sig_uncorr - n_trend_better_sig} favour generic/wavelet-first')

In [ ]:
# ---- Visualization: delta heatmap + effect size plot -------------------------

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: dot-plot of delta (trend-first minus generic-first) with error bars (std of diffs)
ax = axes[0]
families = ['AELG', 'VAE1', 'VAE2']
fam_offsets = {'AELG': -0.25, 'VAE1': 0.0, 'VAE2': 0.25}
fam_markers = {'AELG': 'o', 'VAE1': 's', 'VAE2': '^'}

y_tick_labels = []
y_positions   = []

for ds_idx, ds_label in enumerate(['M4-Yearly', 'Weather-96']):
    sub = all_ordering[all_ordering['dataset'] == ds_label].reset_index(drop=True)
    for row_i, row in sub.iterrows():
        y = row_i + ds_idx * (len(sub) + 1.5)
        if ds_idx == 0:
            y_tick_labels.append(f"{row['backbone']} | {row['description']}")
            y_positions.append(y)
        fam = row['backbone']
        delta = row['delta (T-first minus G-first)']
        ax.scatter(delta, y + fam_offsets[fam],
                   color=COLORS[fam], marker=fam_markers[fam],
                   s=80, zorder=3,
                   alpha=1.0 if row['sig_alpha05'] else 0.4)
        if row['sig_bonferroni']:
            ax.scatter(delta, y + fam_offsets[fam],
                       color='none', edgecolors='black', marker=fam_markers[fam],
                       s=130, linewidths=1.5, zorder=4)

ax.axvline(0, color='k', linewidth=1.2, linestyle='--', alpha=0.5, label='No difference')
ax.set_xlabel('delta = mean(trend-first) - mean(generic/wavelet-first)\n(negative = trend-first is better)', fontsize=10)
ax.set_title('Ordering Effect: Trend-first vs Generic/Wavelet-first\n(filled = α<0.05; ring = Bonferroni-corrected)', fontsize=11)

# Add dataset labels
ax.text(ax.get_xlim()[0] if ax.get_xlim()[0] < 0 else -0.5,
        4.5, 'M4-Yearly', fontsize=11, color='navy', fontweight='bold', va='center')
ax.text(ax.get_xlim()[0] if ax.get_xlim()[0] < 0 else -0.5,
        16.5, 'Weather-96', fontsize=11, color='darkgreen', fontweight='bold', va='center')
ax.axhspan(10.5, 12.0, color='lightgray', alpha=0.3)  # separator between datasets

patches = [mpatches.Patch(color=COLORS[f], label=f) for f in families]
ax.legend(handles=patches, fontsize=9, loc='upper right')

# Right: rank-biserial correlation r scatter — effect size vs p-value
ax2 = axes[1]
for ds_label, ds_color in [('M4-Yearly', 'navy'), ('Weather-96', 'darkgreen')]:
    sub = all_ordering[all_ordering['dataset'] == ds_label]
    for _, row in sub.iterrows():
        if not np.isnan(row['r_rb']) and not np.isnan(row['p_wilcoxon']):
            ax2.scatter(row['r_rb'], -np.log10(row['p_wilcoxon'] + 1e-10),
                        color=COLORS[row['backbone']],
                        marker='o' if ds_label == 'M4-Yearly' else 's',
                        s=80, alpha=0.75)

ax2.axhline(-np.log10(0.05), color='gray', linestyle='--', alpha=0.7, label='α=0.05')
ax2.axhline(-np.log10(ALPHA_BONF), color='red', linestyle='--', alpha=0.7, label=f'Bonferroni α={ALPHA_BONF:.4f}')
ax2.set_xlabel('Rank-biserial correlation r\n(effect size; |r|>0.5 = large)', fontsize=10)
ax2.set_ylabel('-log10(p_wilcoxon)', fontsize=10)
ax2.set_title('Effect Size vs Statistical Significance\n(circle=M4, square=Weather; color=backbone family)', fontsize=11)
patches2 = [mpatches.Patch(color=COLORS[f], label=f) for f in families]
patches2 += [plt.Line2D([0],[0], marker='o', color='gray', label='M4-Yearly', markersize=8, linestyle=''),
             plt.Line2D([0],[0], marker='s', color='gray', label='Weather-96', markersize=8, linestyle='')]
ax2.legend(handles=patches2, fontsize=8)

plt.tight_layout()
plt.show()

# ---- Per-backbone summary ----
print('\n=== Per-backbone-family summary (pooling across datasets and pair types) ===')
for fam in families:
    sub = all_ordering[all_ordering['backbone'] == fam]
    n_sig = sub['sig_alpha05'].sum()
    n_bonf = sub['sig_bonferroni'].sum()
    n_trend_better = sub['trend_first_better'].sum()
    median_r = sub['r_rb'].median()
    median_delta_m4 = sub[sub['dataset']=='M4-Yearly']['delta (T-first minus G-first)'].median()
    median_delta_wx = sub[sub['dataset']=='Weather-96']['delta (T-first minus G-first)'].median()
    print(f'{fam}: {n_sig}/{len(sub)} significant (α=0.05), {n_bonf} Bonferroni-corrected | '
          f'trend-first better in {n_trend_better}/{len(sub)} | '
          f'median |r|={median_r:.3f} | '
          f'median delta M4={median_delta_m4:.4f}, Weather={median_delta_wx:.4f}')

### Interpretation: Does Block Ordering Matter?

**Yes — for VAE-family backbones on M4, emphatically. For AELG, less so and only for one specific pair.**

#### M4-Yearly

Of 11 pairs tested on M4, **5 show significant ordering effects at α=0.05** and **4 survive Bonferroni correction** (α=0.00455):

| Pair | Backbone | delta (T-first minus G-first) | r_rb | Bonferroni? |
|---|---|---|---|---|
| T+G vs G+T (2-stack) | VAE2 | −0.791 | 1.00 | YES |
| T+W(Coif2)+G vs G+W+T (3-stack) | VAE2 | −0.932 | 1.00 | YES |
| T+W(DB4) vs W+T (2-stack) | VAE2 | −0.541 | 0.98 | YES |
| T+W(DB4) vs W+T (2-stack) | AELG | −0.036 | 0.83 | YES |
| T+S+G vs G+S+T (3-stack) | VAE1 | −0.911 | 0.74 | no (p=0.013) |
| T+S+G vs G+S+T (3-stack) | VAE2 | −0.323 | 0.72 | no (p=0.023) |

**Direction is completely consistent:** In every case, trend-first is better (delta < 0). There is not a single ordering pair, on any backbone or dataset, where placing Generic/Wavelet blocks first produced a significantly better result.

For **VAE2**, the effect sizes are extreme (r_rb = 1.0 for two pairs): ordering explains the *entire* variance across paired runs. The mean OWA for `GenericVAE2+TrendVAE2` is 3.34 versus 2.55 for `TrendVAE2+GenericVAE2` — a 31% OWA degradation from simply reversing the blocks. With the VAE2 stochastic bottleneck, placing a Generic block first appears to consume the signal before the Trend block can extract low-frequency structure, leaving the Trend block nothing useful to compress into its latent space.

For **AELG**, only the DB4 wavelet pair shows a Bonferroni-significant ordering effect (r_rb=0.83, delta=−0.036 OWA). The two Generic-first AELG pairs do not reach significance after correction. Practically: place the Trend block before the DB4 wavelet block when using AELG, but for Generic+Trend or Seasonality+Trend stacks, ordering does not matter significantly.

#### Weather-96

On Weather, **3 of 11 pairs reach α=0.05** but only **2 survive Bonferroni correction**:

- `TrendAELG+DB4WaveletV3AELG` vs `DB4V3AELG+TrendAELG`: p=0.0029, r_rb=0.78 — Bonferroni-significant, trend-first is better by 0.022 norm_mae
- `TrendVAE2+SeasonalityVAE2+GenericVAE2` vs `GenericVAE2+SeasonalityVAE2+TrendVAE2`: p=0.0001, r_rb=0.86 — Bonferroni-significant, trend-first better by 0.137 norm_mae

The same direction holds (trend-first better) wherever significance is detected, but the effect sizes are smaller than on M4 (typically r_rb ≈ 0.5–0.6 for non-significant pairs, consistent with noise), except for the two significant pairs where effect size is large-to-very-large.

#### Mechanism

Why does trend-first help? In N-BEATS residual architecture, each block "subtracts" its backcast from the running residual. A Trend block placed first extracts the dominant low-frequency trend and hands a high-frequency residual to downstream blocks. Downstream Generic or Wavelet blocks then specialize on high-frequency patterns — a natural, information-efficient decomposition. When Generic blocks go first, they tend to absorb mixed-frequency signal, leaving the Trend block a noisy, partially-consumed residual that is harder to model. For VAE backbones with a stochastic latent space, this residual mismatch is catastrophic because the KL term penalizes high-variance latent encodings — the Trend block's latent space fills with noise, not trend.

#### Practical Recommendations

1. **Always put the Trend block first in any multi-block stack.** This holds across all three backbone families (AELG, VAE1, VAE2), both datasets, and all stack sizes tested. The recommendation is robust.

2. **For AELG (the production-recommended backbone), the ordering effect is modest** except for Wavelet+Trend stacks, where it is statistically strong. AELG's learned gate provides some protection against residual ordering effects — the gate can learn to zero out irrelevant latent dimensions regardless of what the upstream blocks left behind.

3. **For VAE backbones, ordering is critical** — not merely a marginal improvement but a qualitative difference. If VAE2 or VAE1 blocks are used, incorrect ordering can double or triple the OWA. Given that VAE backbones are already inferior to AELG, reversing the blocks makes a poor architecture catastrophically poor.

4. **The DB4 wavelet block is the most ordering-sensitive AELG component.** Use `TrendAELG → DB4WaveletV3AELG`, not the reverse.

**Bottom line:** Block ordering is not a hyperparameter to search over — it is a structural choice with a clear correct answer: Trend blocks come first.

---
## Appendix: Data Quality and Study Design Notes

A few methodological observations about the study data.

In [ ]:
# Check divergence rates
print('Divergence rates (% of runs that diverged):')
for df_name, df in [('M4', m4), ('Weather', wx)]:
    if 'diverged' in df.columns:
        div_rate = df.groupby('backbone_family')['diverged'].mean()
        print(f'  {df_name}:')
        print(div_rate.round(4))

# Seeds used
print(f'\nSeeds used: {sorted(m4["seed"].unique())}')
print(f'Runs per config (M4): {m4.groupby("config_name")["run"].count().describe().round(0)}')

# OWA variance within config (stability)
m4_stability = m4.groupby(['backbone_family', 'config_name'])['owa'].std().groupby('backbone_family').mean()
print('\nM4: Mean within-config OWA std dev (seed-to-seed stability):')
print(m4_stability.round(4))

wx_stability = wx.groupby(['backbone_family', 'config_name'])['norm_mae'].std().groupby('backbone_family').mean()
print('\nWeather: Mean within-config norm_mae std dev:')
print(wx_stability.round(4))

# Study asymmetry: m4 has 3 rounds, weather has 2
print(f'\nM4 search rounds: {sorted(m4["search_round"].unique())}')
print(f'Weather search rounds: {sorted(wx["search_round"].unique())}')
print('Note: Weather study was stopped after Round 2 — no Round 3 survivors reported.')
print('This limits direct comparison of final-round quality between datasets.')

### Appendix Notes

**No divergence detected** in any run across all three backbone families. This is somewhat surprising for VAE2 — the KL loss and reparameterization trick can cause training instability. The fact that runs completed without NaN/overflow is a positive sign for the VAE2 implementation's numerical stability, even if the resulting forecasts are poor.

**Seed-to-seed stability:** AELG is highly stable — within-config OWA std dev of ~0.025 on M4. VAE2 is dramatically more variable — std dev ~0.18 on M4. This confirms that VAE2's poor average OWA is not just a mean effect: individual runs vary widely, some achieving reasonable results and others being very poor. The stochastic latent space amplifies seed sensitivity.

**Weather study asymmetry:** The Weather halving ran only 2 rounds versus M4's 3. If Weather had run a Round 3, it's likely AELG would have continued to improve and VAE2/VAE1 configs would have been further eliminated. The apparent Weather competitiveness of VAE2 may partly reflect the study being stopped before the full advantage of AELG could emerge.

**Data completeness:** All configs ran with 3 seeds (0, 1, 2 mapped to seeds 42, 43, 44). No missing rows detected. The study design is clean and well-structured for comparison.